In [ ]:
from IPython.display import clear_output
import os

!git clone --branch shantanu https://github.com/AISC-Linear-Probe-Gen/Probe-Generalisation.git
%pip install --upgrade mech-interp-toolkit
clear_output()

os.chdir("/content/Probe-Generalisation/research/obfuscated_activations")

In [ ]:
# Patch simple_ig_with_probes to pass inputs_embeds inside the dict (not as a separate kwarg).
# transformers 5.x @check_model_inputs breaks when nnsight splits inputs_embeds out of the dict
# in tracer.invoke(**synthetic_inputs, inputs_embeds=interpolated_embeddings).
# The working pattern (used in extract_and_cache_activations) is tracer.invoke(**full_dict).

import gc
import torch
from einops import einsum
from mech_interp_toolkit.activation_utils import interpolate_activations, locate_layer_component
from mech_interp_toolkit.gradient_based_attribution import _validate_embeddings, _setup_probe_components, _get_alpha_values, _cleanup_memory
from mech_interp_toolkit.activation_utils import get_embeddings_dict
from mech_interp_toolkit.activation_dict import ActivationDict


def simple_ig_with_probes(model, input_dict, baseline_dict, probe, metric_fn=torch.mean, steps=5):
    """
    Drop-in replacement for mech_interp_toolkit.gradient_based_attribution.simple_ig_with_probes.
    Passes inputs_embeds inside the dict to tracer.invoke (not as a separate kwarg) to avoid
    the transformers 5.x @check_model_inputs / nnsight incompatibility.
    """
    if not torch.is_grad_enabled():
        raise RuntimeError("Integrated Gradients requires gradient computation.")

    input_embeddings = get_embeddings_dict(model, input_dict)["inputs_embeds"]
    baseline_embeddings = get_embeddings_dict(model, baseline_dict)["inputs_embeds"]

    _validate_embeddings(input_embeddings, baseline_embeddings)

    (probe_location, probe_component), probe_weight, probe_bias, scaler = _setup_probe_components(
        probe, input_embeddings
    )

    # Keep attention_mask and any other keys, but NOT inputs_embeds (we'll inject it per step)
    synthetic_inputs = {k: v for k, v in input_dict.items() if k not in ("input_ids", "inputs_embeds")}

    alphas = _get_alpha_values(steps, input_embeddings.dtype)
    accumulated_grads = torch.zeros_like(input_embeddings)

    for alpha in alphas:
        interpolated_embeddings = (
            interpolate_activations(baseline_embeddings, input_embeddings, alpha)
            .detach()
            .requires_grad_(True)
        )

        # Pass inputs_embeds INSIDE the dict — same pattern as get_activations / extract_and_cache_activations
        invoke_inputs = {**synthetic_inputs, "inputs_embeds": interpolated_embeddings}

        with model.trace() as tracer:
            with tracer.invoke(**invoke_inputs):
                acts = locate_layer_component(model, (probe_location, probe_component)).save()
                acts = scaler(acts)
                probe_output = einsum(
                    acts,
                    probe_weight.T,
                    "batch pos d_model, d_model d_probe -> batch pos d_probe",
                ) + probe_bias.view(1, 1, -1)

                if probe.target_type == "classification":
                    if probe_output.shape[-1] == 1:
                        probe_output = torch.sigmoid(probe_output)
                    else:
                        probe_output = torch.softmax(probe_output, dim=-1)
                elif probe.target_type == "regression":
                    pass
                else:
                    raise ValueError(f"Unknown probe target type: {probe.target_type}")

                metric = metric_fn(probe_output)
                if metric.ndim != 0:
                    raise ValueError("Metric function must return a scalar.")
                metric.backward()

        if interpolated_embeddings.grad is None:
            raise RuntimeError("Failed to retrieve gradients.")
        accumulated_grads = accumulated_grads + (interpolated_embeddings.grad / steps)

        _cleanup_memory()

    integrated_grads = ((input_embeddings - baseline_embeddings) * accumulated_grads).sum(dim=-1)
    output = ActivationDict(model.model.config, slice(None), "ig_with_probe_scores")
    output[(0, "layer_in")] = integrated_grads
    model.model.zero_grad(set_to_none=True)

    return output

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Standard library imports
from pathlib import Path
import joblib
import re
from collections import defaultdict

# Third-party imports
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm_notebook as tqdm

# mech_interp_toolkit imports
# Note: simple_ig_with_probes is defined above (cell-1) as a patched version
from mech_interp_toolkit.activation_utils import get_activations, get_embeddings_dict, concat_activations
from mech_interp_toolkit.linear_probes import LinearProbe
from mech_interp_toolkit.utils import load_model_tokenizer_config, set_global_seed

from datasets import load_dataset
from utils.data import extract_user_instruction
import einops

In [15]:
dataset_name = "Mechanistic-Anomaly-Detection/llama3-jailbreaks"
split = "circuit_breakers_test"
model_name = "meta-llama/Llama-3.2-3B-Instruct"
suffix_path = "ra_suffix.pt"
probe_path = Path("t_probes")

batch_size = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

set_global_seed(0)
torch.set_grad_enabled(True)

torch.autograd.grad_mode.set_grad_enabled(mode=True)

In [5]:
class QuickScaler:
    def __init__(self, scaler) -> None:
        self.scale = scaler.scale_
        self.mean = scaler.mean_

    def __call__(self, x) -> torch.Tensor:
        return (x-self.mean)/self.scale


In [9]:
dataset = load_dataset(dataset_name, split=split)
prompts_str = [extract_user_instruction(x) for x in dataset["prompt"]]  # type: ignore
clear_output()

model, ch_tokenizer, config = load_model_tokenizer_config(
    model_name,
    suffix="",
    system_prompt="",
    attn_type="sdpa",
)
clear_output()


def load_suffix(suffix_path: str, device: torch.device) -> torch.Tensor:
    if not suffix_path.endswith(".pt"):
        raise ValueError("suffix_path must be a .pt embedding file")
    suffix_emb = torch.load(suffix_path, map_location=device)
    if suffix_emb.dim() == 2:
        suffix_emb = suffix_emb.unsqueeze(0)
    return suffix_emb


n_layers = config.num_hidden_layers
components = [(i, "layer_out") for i in range(n_layers)]

suffix_embed = load_suffix(suffix_path, device)
len_suffix = suffix_embed.shape[1]

In [10]:
probe_dict = {}
for f in probe_path.glob("*.joblib"):
    layer = int(re.findall(r"layer_(\d+)", str(f))[0])
    probe = joblib.load(f)

    lp = LinearProbe("classification")
    lp.location = (layer, "layer_out")
    lp.weight = probe["model"].coef_
    lp.bias = probe["model"].intercept_
    lp.scaler = QuickScaler(probe["scaler"])  # type:ignore

    probe_dict[layer] = lp

In [11]:
def metric_fn(x):
    return x[:,-1,:]

outputs = defaultdict(list)

In [ ]:
for layer in range(n_layers):

    for batch_start in tqdm(range(0, len(prompts_str), batch_size), desc=str(f"layer: {layer}")):
        batch_prompts = prompts_str[batch_start : batch_start + batch_size]

        batch_dict = ch_tokenizer(prompts=batch_prompts)
        batch_embeds_dict = get_embeddings_dict(model, batch_dict)
        batch_embeds = batch_embeds_dict["inputs_embeds"]
        batch_attn_mask = batch_embeds_dict["attention_mask"]

        batch_suffix = einops.repeat(
                suffix_embed,
                "1 pos d_model -> b pos d_model",
                b=len(batch_prompts),
            )
        new_embeds_dict = {
            "inputs_embeds": torch.cat(
                [batch_embeds[:, :-5, :], batch_suffix, batch_embeds[:, -5:, :]], dim=1
            ),
            "attention_mask": torch.cat(
                [batch_attn_mask, torch.ones(len(batch_prompts), len_suffix, device=batch_attn_mask.device)],
                dim=1,
            ),
        }

        baseline = {
            "inputs_embeds": torch.zeros_like(new_embeds_dict["inputs_embeds"]),
            "attention_mask": torch.cat(
                [batch_attn_mask, torch.ones(len(batch_prompts), len_suffix, device=batch_attn_mask.device)],
                dim=1,
            ),
        }

        ig_scores = simple_ig_with_probes(
            model=model,
            input_dict=new_embeds_dict,
            baseline_dict=baseline,
            probe=probe_dict[layer]
        )

        outputs[layer].append(ig_scores)

In [14]:
new_embeds_dict

{'inputs_embeds': tensor([[[-1.5137e-02,  1.0071e-02, -4.9438e-03,  ...,  3.7689e-03,
           -1.1475e-02, -2.3346e-03],
          [-1.5137e-02,  1.0071e-02, -4.9438e-03,  ...,  3.7689e-03,
           -1.1475e-02, -2.3346e-03],
          [-1.5137e-02,  1.0071e-02, -4.9438e-03,  ...,  3.7689e-03,
           -1.1475e-02, -2.3346e-03],
          ...,
          [ 1.5442e-02, -2.3193e-02,  1.5869e-02,  ..., -2.5940e-03,
            5.8594e-03, -6.0791e-02],
          [-1.1658e-02,  6.1646e-03,  5.2185e-03,  ...,  9.5825e-03,
           -1.6708e-03, -1.6235e-02],
          [-6.5308e-03, -6.9809e-04, -6.7902e-04,  ...,  4.3869e-04,
            1.4038e-02,  1.0834e-03]],
 
         [[-1.5137e-02,  1.0071e-02, -4.9438e-03,  ...,  3.7689e-03,
           -1.1475e-02, -2.3346e-03],
          [-1.5137e-02,  1.0071e-02, -4.9438e-03,  ...,  3.7689e-03,
           -1.1475e-02, -2.3346e-03],
          [-1.5137e-02,  1.0071e-02, -4.9438e-03,  ...,  3.7689e-03,
           -1.1475e-02, -2.3346e-03],
  

In [ ]:
import pickle

with open("sig_scores.pkl", "wb") as f:
    pickle.dump(outputs, f)